# 🧪 RontoGPT — 4 adaptateurs **distillés** pour Gemma-3-1B

Ce notebook entraîne les 4 adaptateurs LoRA du projet (`aura-web`, `aura-math`,
`aura-code`, `aura-general`) sur des données **filtrées**, avec le **template
réel de Gemma-3** et le prompt système français cible.

Runtime → *Modifier le type d'exécution* → **T4 GPU**, puis *Exécuter tout*
(~20-25 min).

## Ce que ce notebook corrige dans le plan initial

| Plan initial | Problème vérifié | Ici |
|---|---|---|
| `<|im_start|>user…<|im_end|>` | Ce n'est **pas** le template de Gemma-3 (c'est du ChatML, format Qwen). Le GGUF du projet déclare `<bos><start_of_turn>user…<end_of_turn>` et préfixe le message système au **premier tour utilisateur**. | `tokenizer.apply_chat_template()` : le template vient du modèle, pas d'une chaîne recopiée |
| `iamtarun/python_code_instructions_20k`, `bofenghuang/viggo-fr`, `polyglot-data/magpie-french-ultra` | **Introuvables** sur Hugging Face : l'ID exact ne répond pas, et la recherche ne trouve que `CATIE-AQ/viggo_french`, une autre traduction (vérifié le 24 oct. 2026). | 3 sources **vérifiées** : `AgentPublic/piaf` (FR), `TIGER-Lab/MathInstruct`, `sahil2801/CodeAlpaca-20k` |
| `filtrer_et_sauvegarder(..., "instruction")` puis `{"text": texte}` | Garde **l'instruction seule** : aucune paire question/réponse, donc rien à apprendre. | Chaque exemple est une paire `question` → `reponse` (avec `contexte` pour le RAG) |
| « température 1.0 » | Un LoRA ne « réagit » pas à la température : elle ne s'applique qu'à l'échantillonnage, à l'inférence. | Le LoRA apprend une distribution ; la température reste un réglage d'inférence |
| « routeur MoE Top-2 par token » | Les échelles LoRA s'appliquent **par requête** dans llama-server ; un routage par token exige un vrai modèle MoE. | Rien à faire ici : `serveur_lora` route en Top-2 **par requête** |
| Dépôt dans `adaptateurs/` | Le chemin serveur **actif** (`AURA_SERVEUR=1`) lit `serveur/aura-*.gguf` ; `adaptateurs/` ne sert qu'au hot-swap in-process. | Export dans les **deux** dossiers, avec la bonne priorité expliquée à la fin |

## Ce qui améliore réellement la qualité

1. **Template exact** du cerveau (lu depuis le tokenizer) ;
2. **masquage du prompt** : la perte ne porte que sur la réponse, pas sur la consigne ;
3. cibles LoRA **attention + MLP** (`q/k/v/o` + `gate/up/down`) ;
4. filtre qualité + déduplication + plafond de longueur ;
5. **jeu de validation** (5 %) : `eval_loss` et générations de contrôle ;
6. prompt système **cible** (`SYSTEME_FR`) : l'adaptateur apprend à le suivre
   au lieu de le recopier, ce que le 1B de base fait (mesuré : 10/14 au banc).

## 1. Dépendances

`torchao` embarqué par Colab est trop ancien pour `peft` (>0.16 requis) et le
LoRA ne l'utilise pas : on le retire. `sentencepiece` est requis par le
tokenizer Gemma.

In [ ]:
!pip -q uninstall -y torchao
!pip -q install -U datasets peft transformers accelerate gguf sentencepiece

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "AUCUN — active le T4 avant d'entraîner")
print("bf16 :", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

## 2. Réglages et prompt système

`SYSTEME` est la copie exacte de `aura/prompt_systeme.SYSTEME_FR` (source :
`aura/prompt_systeme.py`). Les adaptateurs sont entraînés **avec** ce prompt :
c'est ce qui leur apprend à le respecter sans le recopier. À l'inférence, il
faut donc lancer avec `AURA_SYSTEME=fr` (ou `[llama] systeme = "fr"`).

In [ ]:
BASE = "unsloth/gemma-3-1b-it"   # même architecture que le GGUF Q8 du projet
N_MAX = 300                      # cible PAR CATÉGORIE (le filtre peut rendre moins)
GRAINE = 42
MAX_LENGTH = 640
CATEGORIES = ["web", "math", "code", "general"]

SYSTEME = """<role>
Tu es RontoGPT, un assistant francophone précis, direct et honnête. Tu réponds en français, sauf si l'utilisateur demande explicitement une autre langue.
</role>

<priorites>
1. Exactitude : une réponse juste et vérifiable passe avant une réponse longue.
2. Ancrage : quand un CONTEXTE ou des FAITS sont fournis, n'utilise QUE ces informations ; si la réponse n'y figure pas, dis-le en une phrase au lieu d'inventer.
3. Concision : pas de préambule, pas de reformulation de la question, pas de conclusion de remplissage.
</priorites>

<contraintes>
- Nous sommes en 2026 : pour toute question sensible au temps, utilise cette année.
- N'invente ni chiffre, ni date, ni citation, ni source.
- Si la demande est ambiguë, pose UNE question de clarification courte plutôt que de deviner.
- Ne montre pas ton raisonnement interne : donne seulement la réponse finale, sauf si l'utilisateur demande les étapes.
</contraintes>

<format>
- Question simple : 1 à 3 phrases.
- Explication ou comparaison : paragraphes courts ou liste de points.
- Code : bloc exécutable, sans commentaire décoratif.
- Analyse ou dissertation : introduction, 2 à 3 parties, conclusion.
</format>

<methode>
Analyse la demande, réponds, puis relis-toi avant d'envoyer. Ces étapes restent internes.
</methode>"""

print(f"{len(SYSTEME)} caractères de prompt système")

## 3. Données : locales vérifiées + 3 sources Hugging Face vérifiées

Les données **locales** du projet (`datasets/*.jsonl`, françaises et vérifiées)
passent en premier et ne sont jamais écartées par le tirage. Les sources
externes ne servent qu'à compléter :

| Catégorie | Source | Langue | Champs utilisés |
|---|---|---|---|
| `web` | `AgentPublic/piaf` (`plain_text`) | français | `question`, `answers.text[0]`, `context` |
| `math` | `TIGER-Lab/MathInstruct` (`default`) | anglais (surtout symbolique) | `instruction`, `output` |
| `code` | `sahil2801/CodeAlpaca-20k` (`default`) | anglais | `instruction` + `input`, `output` |
| `general` | locales + `datasets/general_amplifie.jsonl` (~184) | français | `question`, `reponse` |

Pourquoi pas `argilla/magpie-ultra-v0.1` : le jeu existe, mais il est
**anglais** (licence Llama 3.1) ; l'utiliser pour l'adaptateur « style
français » apprendrait exactement ce qu'on veut éviter. Pour `general`, moins
de 300 exemples français valent mieux que 300 exemples anglais.

Le filtre est volontairement strict : longueur, refus, marqueurs de template
étranger, URL, doublons. Il **rapporte** ce qu'il écarte.

In [ ]:
# 3. Construction des datasets filtrés
import json, random, re, subprocess, unicodedata
from pathlib import Path

RACINE = Path("/content")
SORTIE = RACINE / "datasets_distilles"
SORTIE.mkdir(exist_ok=True)
random.seed(GRAINE)

# id, config, format de lecture — vérifiés via l'API Hugging Face le 24/10/2026
SOURCES = {
    "web": [("AgentPublic/piaf", "plain_text", "piaf")],
    "math": [("TIGER-Lab/MathInstruct", "default", "instruction_output")],
    "code": [("sahil2801/CodeAlpaca-20k", "default", "instruction_input_output")],
    "general": [],          # local seulement (Magpie Ultra est en anglais)
}

LIMITES = {"web": 420, "math": 380, "code": 520, "general": 420}
FR_INDICES = {"le", "la", "les", "des", "une", "un", "est", "et", "pour",
              "que", "qui", "dans", "sur", "avec", "pas", "ce", "cette",
              "au", "aux", "du", "en"}


def _sans_accent(t):
    return "".join(c for c in unicodedata.normalize("NFD", t or "")
                   if unicodedata.category(c) != "Mn")


def _francais(*textes):
    mots = re.findall(r"[a-z]+", _sans_accent(" ".join(textes).lower()))
    return sum(m in FR_INDICES for m in mots) >= 3


def _tokens(t):
    # approximation volontairement simple : 1 mot ≈ 1,3 token
    return len(t.split()) * 1.3


def _garde(q, r, cat):
    q, r = (q or "").strip(), (r or "").strip()
    if not (12 <= len(q) <= 700) or not (15 <= len(r) <= 3000):
        return False
    if _tokens(q + " " + r) > LIMITES[cat]:
        return False
    for mauvais in ("<|im_start|>", "<|im_end|>", "<start_of_turn>",
                    "<end_of_turn>", "As an AI", "I cannot",
                    "Lorem ipsum", "http://", "https://"):
        if mauvais in q + r:
            return False
    if cat in ("web", "general") and not _francais(q, r):
        return False
    return True


def _paire(brut, cat):
    """(question, réponse, contexte) depuis un enregistrement HF."""
    if "question" in brut and "answers" in brut:          # Piaf
        ans = brut["answers"]
        textes = ans.get("text") if isinstance(ans, dict) else ans
        return (brut.get("question", ""), textes[0] if textes else "",
                brut.get("context", ""))
    q = brut.get("instruction") or brut.get("question") or ""
    r = brut.get("output") or brut.get("response") or ""
    entree = brut.get("input") or ""
    if entree.strip() not in ("", "< noinput >", "<noinput>"):
        q = f"{q}\n\nEntrée :\n{entree}"
    return q, r, ""


def _depuis_local(cat):
    chemins = [RACINE / "aura-1b" / "datasets" / f"{cat}.jsonl"]
    if cat == "general":
        # questions reformulees par scripts/amplifier_general.py (reponses
        # inchangees) : present dans le depot si le script a ete lance.
        chemins.append(RACINE / "aura-1b" / "datasets" / "general_amplifie.jsonl")
    chemins = [c for c in chemins if c.is_file()]
    if not chemins:
        return [], 0
    lignes = []
    for chemin in chemins:
        lignes += chemin.read_text(encoding="utf-8").splitlines()
    gardes, vus = [], set()
    for ligne in lignes:
        if not ligne.strip():
            continue
        e = json.loads(ligne)
        q, r = e.get("question", ""), e.get("reponse", "")
        cle = _sans_accent(q.lower())
        if _garde(q, r, cat) and cle not in vus:
            vus.add(cle)
            gardes.append({"question": q, "reponse": r,
                           **({"contexte": e["contexte"]} if e.get("contexte") else {})})
    return gardes, len(gardes)


def _depuis_hf(cat, vus):
    from datasets import load_dataset
    gardes = []
    for repo, config, _format in SOURCES[cat]:
        print(f"  {repo} ...", end=" ")
        avant = len(gardes)
        for i, brut in enumerate(load_dataset(repo, config, split="train",
                                              streaming=True)):
            if i >= 20000:              # borne le scan : rendement décroissant
                break
            q, r, ctx = _paire(brut, cat)
            cle = _sans_accent(q.lower())
            if not _garde(q, r, cat) or cle in vus:
                continue
            vus.add(cle)
            gardes.append({"question": q, "reponse": r,
                           **({"contexte": ctx} if ctx else {})})
            if len(gardes) - avant >= N_MAX * 4:
                break
        print(f"{len(gardes) - avant} gardés")
    return gardes


def construire(cat):
    local, n_local = _depuis_local(cat)
    vus = {_sans_accent(e["question"].lower()) for e in local}
    hf = _depuis_hf(cat, vus)
    exemples = local[:N_MAX]
    manque = N_MAX - len(exemples)
    if manque > 0 and hf:
        exemples += random.sample(hf, min(manque, len(hf)))
    random.shuffle(exemples)
    chemin = SORTIE / f"{cat}.jsonl"
    with chemin.open("w", encoding="utf-8") as f:
        for e in exemples:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")
    print(f"{cat:8s} {len(exemples):4d} exemples "
          f"(local {n_local}, HF {len(exemples) - n_local})")
    return len(exemples)


# les données locales viennent du dépôt ; le notebook tourne quand même sans.
if not (RACINE / "aura-1b").is_dir():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Simonc44/aura-1b.git",
                    str(RACINE / "aura-1b")], check=False)

compte = {cat: construire(cat) for cat in CATEGORIES}
(SORTIE / "compte.json").write_text(json.dumps(compte, indent=2), encoding="utf-8")
print("\nTotal :", compte)

## 4. Formatage : le template du modèle, et la perte sur la réponse seule

Deux détails font la différence entre un adaptateur qui « parle » et un
adaptateur qui **répond** :

1. `apply_chat_template` produit `<bos><start_of_turn>user\n…<end_of_turn>\n<start_of_turn>model\n…`
   — exactement ce que llama-server enverra à l'inférence ;
2. les tokens du prompt (système + question) sont mis à `-100` : la perte ne
   porte que sur la réponse. Sans ce masquage, le modèle dépense sa capacité à
   recopier la consigne.

In [ ]:
# 4. Dataset tokenisé, prompt masqué
import torch
from torch.utils.data import Dataset as TorchDataset
from transformers import AutoTokenizer


def charger_tokenizer():
    tok = AutoTokenizer.from_pretrained(BASE)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


class Exemples(TorchDataset):
    def __init__(self, chemin, tok):
        self.exemples = []
        lignes = [json.loads(l) for l in Path(chemin).read_text(encoding="utf-8").splitlines()
                  if l.strip()]
        for e in lignes:
            enc = self._encoder(e, tok)
            if enc is not None:
                self.exemples.append(enc)
        print(f"  {len(self.exemples)}/{len(lignes)} exemples formatés")

    @staticmethod
    def _encoder(e, tok):
        contenu = e["question"]
        if e.get("contexte"):
            contenu = f"CONTEXTE :\n{e['contexte']}\n\nQUESTION :\n{contenu}"
        conv = [{"role": "system", "content": SYSTEME},
                {"role": "user", "content": contenu},
                {"role": "model", "content": e["reponse"]}]
        complet = tok.apply_chat_template(conv, tokenize=False)
        invite = tok.apply_chat_template(conv[:-1], tokenize=False,
                                         add_generation_prompt=True)
        if not complet.startswith(invite):
            return None                     # template inattendu : on n'invente pas
        ids = tok(complet, add_special_tokens=False)["input_ids"]
        n_prompt = len(tok(invite, add_special_tokens=False)["input_ids"])
        if n_prompt >= MAX_LENGTH - 8:
            return None                     # réponse entièrement tronquée
        ids = ids[:MAX_LENGTH]
        labels = [-100] * n_prompt + ids[n_prompt:]
        if not any(l != -100 for l in labels):
            return None
        return {"input_ids": ids, "labels": labels,
                "attention_mask": [1] * len(ids), "question": contenu}

    def __len__(self):
        return len(self.exemples)

    def __getitem__(self, i):
        e = self.exemples[i]
        return {k: e[k] for k in ("input_ids", "labels", "attention_mask")}


def collate(batch, pad_id):
    n = max(len(x["input_ids"]) for x in batch)
    ids, labels, att = [], [], []
    for x in batch:
        d = n - len(x["input_ids"])
        ids.append(x["input_ids"] + [pad_id] * d)
        att.append(x["attention_mask"] + [0] * d)
        labels.append(x["labels"] + [-100] * d)
    return {"input_ids": torch.tensor(ids),
            "attention_mask": torch.tensor(att),
            "labels": torch.tensor(labels)}


# contrôle immédiat sur un exemple réel
_tok = charger_tokenizer()
_demo = Exemples(SORTIE / f"{CATEGORIES[0]}.jsonl", _tok)
_ex = _demo[0]
print("tokens réponse :", sum(1 for l in _ex["labels"] if l != -100))
print("début du texte :", _tok.decode(_ex["input_ids"])[:180].replace("\n", " | "))

## 5. Entraînement (PEFT, attention + MLP)

Pourquoi PEFT et pas Unsloth : le dépôt n'a **aucune** dépendance Unsloth, et
la conversion GGUF existante (`convert_lora_to_gguf.py`) est déjà celle qui
produit les adaptateurs actuels. On garde donc la chaîne éprouvée ; Unsloth
n'apporterait que de la vitesse.

Cibles LoRA : `q/k/v/o` **et** `gate/up/down` — les MLP portent une grande
partie du style et des connaissances ; l'ancien notebook ne ciblait que
l'attention.

In [ ]:
# 5. Boucle d'entraînement (une catégorie = un adaptateur)
import gc, inspect
from peft import LoraConfig, get_peft_model
from transformers import (AutoModelForCausalLM, Trainer, TrainingArguments)


def _dtype(torch):
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def _eval_kwarg():
    params = inspect.signature(TrainingArguments.__init__).parameters
    return "eval_strategy" if "eval_strategy" in params else "evaluation_strategy"


def entrainer(cat, epochs=3):
    final = Path(f"tmp-lora-{cat}/final")
    if (final / "adapter_config.json").is_file():
        print(f"[SKIP] {cat} : adaptateur déjà présent")
        return final
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    tok = charger_tokenizer()
    ds = Exemples(SORTIE / f"{cat}.jsonl", tok)
    if len(ds) < 20:
        print(f"[{cat}] seulement {len(ds)} exemples : à compléter avant d'entraîner")
        return None
    n_val = max(1, len(ds) // 20)
    train, val = torch.utils.data.random_split(
        ds, [len(ds) - n_val, n_val],
        generator=torch.Generator().manual_seed(GRAINE))

    modele = AutoModelForCausalLM.from_pretrained(
        BASE, torch_dtype=_dtype(torch), device_map="auto")
    config = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM")
    modele = get_peft_model(modele, config)
    modele.print_trainable_parameters()

    args = TrainingArguments(
        output_dir=f"tmp-lora-{cat}",
        num_train_epochs=epochs,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        fp16=not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
        save_strategy="no",
        logging_steps=10,
        report_to=[],
        seed=GRAINE,
        **{_eval_kwarg(): "epoch"},
    )
    trainer = Trainer(model=modele, args=args, train_dataset=train,
                      eval_dataset=val,
                      data_collator=lambda b: collate(b, tok.pad_token_id))
    trainer.train()
    metriques = trainer.evaluate()
    print(f"[{cat}] eval_loss = {metriques.get('eval_loss', float('nan')):.4f}")

    # contrôle qualitatif : 2 générations sur des exemples vus à l'entraînement
    modele.eval()
    for enc in ds.exemples[-2:]:
        conv = [{"role": "system", "content": SYSTEME},
                {"role": "user", "content": enc["question"]}]
        invite = tok.apply_chat_template(conv, tokenize=False,
                                         add_generation_prompt=True)
        entree = tok(invite, return_tensors="pt",
                     add_special_tokens=False).to(modele.device)
        with torch.no_grad():
            sortie = modele.generate(**entree, max_new_tokens=100, do_sample=False)
        texte = tok.decode(sortie[0][entree["input_ids"].shape[1]:],
                           skip_special_tokens=True)
        print("  →", texte[:220].replace("\n", " "))

    modele.save_pretrained(final)
    del trainer, modele
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return final


resultats = {}
for cat in CATEGORIES:
    print("\n" + "=" * 60 + f"\n[CAT] {cat}\n" + "=" * 60)
    resultats[cat] = entrainer(cat)
print("\nAdaptateurs :", resultats)

## 6. Conversion PEFT → GGUF

`convert_lora_to_gguf.py` écrit un fichier que `llama-server --lora` sait
monter. Sortie **q8_0**, comme les adaptateurs actuels.

In [ ]:
# 6. Conversion en GGUF LoRA
!git clone --depth 1 --filter=blob:none --sparse https://github.com/ggml-org/llama.cpp.git tmp-llama-cpp
%cd tmp-llama-cpp
!git sparse-checkout set conversion gguf-py --skip-checks
%cd /content

import subprocess
from pathlib import Path

Path("adaptateurs").mkdir(exist_ok=True)
env = dict(os.environ, PYTHONPATH="tmp-llama-cpp/gguf-py")
for cat in CATEGORIES:
    final = Path(f"tmp-lora-{cat}/final")
    if not (final / "adapter_config.json").is_file():
        print(f"[{cat}] ignoré : adaptateur absent")
        continue
    cmd = ["python", "tmp-llama-cpp/convert_lora_to_gguf.py",
           "--outfile", f"adaptateurs/aura-{cat}.gguf",
           "--outtype", "q8_0", "--base-model-id", BASE, str(final)]
    r = subprocess.run(cmd, env=env, capture_output=True, text=True)
    print(f"[{cat}]", "OK" if r.returncode == 0 else r.stderr[-800:])
!ls -lh adaptateurs/*.gguf

## 7. Livraison et montage dans RontoGPT

Deux dossiers, deux chemins :

| Dossier | Chemin | Quand |
|---|---|---|
| `serveur/aura-*.gguf` | **serveur llama-server** (`AURA_SERVEUR=1`, défaut du projet) | les 4 adaptateurs sont montés **au boot** du serveur |
| `adaptateurs/aura-*.gguf` | hot-swap in-process (`AURA_ADAPTATEURS=1`) | repli sans serveur |

Après le téléchargement :

1. dépose les 4 `.gguf` dans `serveur/` sur le PC ;
2. **redémarre** le serveur (il monte les LoRA au démarrage) ;
3. vérifie : `curl http://127.0.0.1:8757/lora-adapters` doit lister 4 entrées ;
4. lance avec le prompt cible : `AURA_SYSTEME=fr` (ou `[llama] systeme = "fr"`).

Sans `AURA_SYSTEME=fr`, l'adaptateur est monté mais reçoit l'ancien prompt
anglais : il répond, mais il n'est pas dans les conditions de son entraînement.

In [ ]:
# 7. Zip final (serveur/ + adaptateurs/)
import shutil
from pathlib import Path

Path("serveur").mkdir(exist_ok=True)
for gguf in Path("adaptateurs").glob("aura-*.gguf"):
    shutil.copy2(gguf, Path("serveur") / gguf.name)

!zip -j adaptateurs_aura_gemma3.zip adaptateurs/aura-*.gguf
!cd /content && cp adaptateurs_aura_gemma3.zip serveur_aura_gemma3.zip && ls -lh serveur_aura_gemma3.zip

from google.colab import files
files.download("serveur_aura_gemma3.zip")

## 8. Contrôles et limites

- **`eval_loss` monte d'une époque à l'autre** → surapprentissage sur 300
  exemples : réduis `epochs` à 2, ou ajoute des données. Ne baisse pas le
  `lora_dropout` pour « compenser ».
- **Les générations de contrôle recopient la consigne** → le prompt est
  encore trop long pour la capacité restante : vérifie que le masquage est
  actif (`tokens réponse` > 0) et que `SYSTEME` est bien celui utilisé à
  l'inférence.
- **Les adaptateurs ne sont pas un MoE** : ils ne s'activent pas par token.
  Le routeur Top-2 de `serveur_lora` choisit **une paire d'experts par
  requête** et renormalise leurs échelles.
- **Un LoRA n'ajoute pas de connaissances fiables** : il déplace le style et
  les réflexes. Les faits viennent du RAG, du graphe et des experts exacts.